คูณ 1000

In [ ]:
import pandas as pd

df = pd.read_csv('original.csv')
df['Total'] = df['Consumption'] * 1000

df.head()
df.to_csv('final.csv', index=False)

แปลงไฟล์

In [ ]:
import pandas as pd

# โหลดไฟล์ CSV (เปลี่ยนชื่อไฟล์ตามที่คุณมี)
df = pd.read_csv("R1 cutting.csv", encoding="utf-8-sig", header=None)

# สร้างลิสต์เก็บผลลัพธ์
paired_results = []

# ใช้การวนลูปแบบกลุ่มละ 3 แถว: PEAK, โรงงาน, มิเตอร์
i = 0
while i < len(df) - 2:
    row_peak = df.iloc[i]
    row_factory = df.iloc[i + 1]
    row_meter = df.iloc[i + 2]

    # ตรวจว่าตรงโครงสร้างที่ต้องการหรือไม่
    if (
        str(row_peak[1]).strip() == "PEAK" and
        pd.notna(row_peak[2]) and
        pd.notna(row_factory[0]) and
        pd.notna(row_meter[0]) and
        str(row_meter[1]).strip() == "OFF-PEAK"
    ):
        try:
            factory = str(row_factory[0]).strip()
            meter = str(row_meter[0]).strip()
            peak_units = float(str(row_peak[2]).replace(",", "").strip())
            offpeak_units = float(str(row_meter[2]).replace(",", "").strip())

            full_name = f"{factory}-{meter}"

            paired_results.append({
                "PrintingName": full_name,
                "PeriodText": "PEAK",
                "Total": peak_units
            })

            paired_results.append({
                "PrintingName": full_name,
                "PeriodText": "OFF-PEAK",
                "Total": offpeak_units
            })
        except:
            pass

        i += 3  # ข้ามไปกลุ่มใหม่
    else:
        i += 1  # ไม่ตรงรูปแบบ ก็เลื่อนไปแถวถัดไป

# สร้าง DataFrame และบันทึกไฟล์
df_result = pd.DataFrame(paired_results)
df_result.to_csv("โรงงานมิเตอร์-รวมPEAK-OFFPEAK.csv", index=False, encoding="utf-8-sig")

# แสดงผลลัพธ์บางส่วน
print(df_result.head())


เปรียบเทียบสองไฟล์

In [ ]:
import pandas as pd
import numpy as np

FILE_A = "final.csv"
FILE_B = "โรงงานมิเตอร์-รวมPEAK-OFFPEAK.csv"
ENCODING = "utf-8-sig"

KEY_COLS = ["PrintingName", "PeriodText"]
TOTAL_COL = "Total"
TOLERANCE = 0.01

dfA = pd.read_csv(FILE_A, encoding=ENCODING)
dfB = pd.read_csv(FILE_B, encoding=ENCODING)

for df in (dfA, dfB):
    for col in KEY_COLS:
        df[col] = (
            df[col]
              .astype(str)
              .str.strip()
              .str.upper()
        )

m = (
    dfA[KEY_COLS + [TOTAL_COL]]
        .rename(columns={TOTAL_COL: "Total_A"})
        .merge(
            dfB[KEY_COLS + [TOTAL_COL]].rename(columns={TOTAL_COL: "Total_B"}),
            on=KEY_COLS,
            how="outer",
            indicator=True
        )
)

m["is_mismatch"] = (
    (m["_merge"] == "both") &
    ~np.isclose(m["Total_A"], m["Total_B"], atol=TOLERANCE, rtol=0)
)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

diff_rows = (
    m.loc[m["is_mismatch"], KEY_COLS + ["Total_A", "Total_B"]]
     .assign(Diff=lambda d: d["Total_A"] - d["Total_B"])
     .sort_values(KEY_COLS)
)

only_A = m.loc[m["_merge"] == "left_only", KEY_COLS + ["Total_A"]]
only_B = m.loc[m["_merge"] == "right_only", KEY_COLS + ["Total_B"]]

print("\n===  Total ไม่ตรงกัน (เกิน ±{:.2f}) ===".format(TOLERANCE))
print(diff_rows if not diff_rows.empty else "• (ไม่พบ)")

print("\n===  อยู่ในไฟล์ A แต่ไม่มีใน B ===")
print(only_A if not only_A.empty else "• (ไม่พบ)")

print("\n===  อยู่ในไฟล์ B แต่ไม่มีใน A ===")
print(only_B if not only_B.empty else "• (ไม่พบ)")

diff_rows.to_csv("total_mismatch.csv", index=False)
only_A.to_csv("only_in_A.csv", index=False, encoding=ENCODING)
only_B.to_csv("only_in_B.csv", index=False, encoding=ENCODING)